In [2]:
%pip install simfile pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 29.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 47.4 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import glob

# packs from my setup
songs_path = '/media/win/Users/Lenovo/AppData/Roaming/ITGmania/Songs/'
pack_names = [
    'Stamina RPG 6',
    'SRPG6 NNP',
    'Stamina RPG 7',
    'Stamina RPG 7 - SN',
    'Stamina RPG 7 - FE',
    'Stamina RPG 8',
    'Stamina RPG 8 - SN',
    'Stamina RPG 8 - FE',
    'Stamina RPG 9',
    'Stamina RPG 9 - SN',
    'Stamina RPG 9 - FE',
    'Stamina RPG 10',
    'Stamina RPG 10 - SN',
    'Stamina RPG 10 - FE',
]
pack_paths = [os.path.join(songs_path, name) for name in pack_names]

glob_path = './packs/*'
pack_paths += glob.glob(glob_path)

In [9]:
from simfile.dir import SimfilePack
from analysis.analyzer import SongAnalyzer
from analysis.hash import get_hash
import pandas as pd

try:
    existing_df = pd.read_json('../public/dataset.json')
    seen_hashes = set(existing_df['hash'])
except FileNotFoundError:
    existing_df = None
    seen_hashes = set()

data = []
for pack_path in pack_paths:
    pack = SimfilePack(pack_path)
    for song in pack.simfiles(strict=False):
        song_analyzer = SongAnalyzer(song)
        song_str = f'{pack.name}/{song.title or ""}'

        for chart_analyzer in song_analyzer.chart_analyzers.values():
            chart = chart_analyzer.chart
            if (chart.stepstype or '').lower() != 'dance-single':
                continue
            diff = (chart.difficulty or '').lower()
            title = song_str
            if diff != 'challenge':
                title += f' ({diff})'

            chart_hash = get_hash(song, chart_analyzer.chart)
            if chart_hash in seen_hashes:
                print(f'skipping {title} (dupe hash)')
                continue
            seen_hashes.add(chart_hash)
            print(f'analyzing {title}')

            stream_info = chart_analyzer.get_stream_info()
            total_stream = stream_info['total_stream']
            total_break = stream_info['total_break']
            bpms = stream_info['bpms']
            avg_bpm = stream_info['avg_bpm']
            try:
                meter = int(chart_analyzer.chart.meter)
            except ValueError:
                print(song.title, ': invalid meter')
                continue

            if total_stream == 0:
                print(title, ': no stream')
            else:
                mult = stream_info['quant'] / 16
                data.append((
                    title,
                    chart_hash,
                    meter,
                    round(avg_bpm * mult, 3),
                    total_stream * mult,
                    total_stream / (total_stream + total_break),
                    stream_info['segments']
                ))

skipping Stamina RPG 6/[11] [128] Alcohol-Free (dupe hash)
skipping Stamina RPG 6/[11] [136] Cowboy (dupe hash)
skipping Stamina RPG 6/[11] [136] Flutterwonder (Hard) (dupe hash)
skipping Stamina RPG 6/[11] [124] Glory Days (dupe hash)
skipping Stamina RPG 6/[11] [145] Hawaiian Roller Coaster Ride (Medium) (dupe hash)
skipping Stamina RPG 6/[11] [131] In The Game (The Football Song) (dupe hash)
skipping Stamina RPG 6/[11] [145] Kislotnyy Dozhd (Medium) (dupe hash)
skipping Stamina RPG 6/[11] [120] Le Perv (dupe hash)
skipping Stamina RPG 6/[11] [124] Lonely (dupe hash)
skipping Stamina RPG 6/[11] [145]  Move That Body! (dupe hash)
skipping Stamina RPG 6/[11] [147] This game (Hard) (dupe hash)
skipping Stamina RPG 6/[11] [128] We Are Giants (feat. Dia Frampton) (dupe hash)
skipping Stamina RPG 6/[12] [138] Cake By The Ocean (dupe hash)
skipping Stamina RPG 6/[12] [174] Gargantua (dupe hash)
skipping Stamina RPG 6/[12] [160] Gimme Your Desire (Hard) (dupe hash)
skipping Stamina RPG 6/[12

In [10]:
columns = ['name', 'hash', 'meter', 'bpm', 'totalStream', 'streamDensity', 'segments']
df = pd.DataFrame(data, columns=columns)
if existing_df is not None:
    df = pd.concat([existing_df, df], axis=0, ignore_index=True)
df

,name,hash,meter,bpm,totalStream,streamDensity,segments
0,Stamina RPG 6/[11] [128] Alcohol-Free,0b020e29c3919c10c1d3fb2776e813b18af30114,11,128.000,52.0,0.626506,"[3, 8, -8, 15, -5, 3, -9, 8, -8, 15]"
1,Stamina RPG 6/[11] [136] Cowboy,f2bc769bdf7addfc64d7e46643e194f184820a56,11,136.000,33.0,0.492537,"[7, -17, 7, -16, 2, 17]"
2,Stamina RPG 6/[11] [136] Flutterwonder (Hard),fc05556c0b501fabafe79a16923e910518d5d03d,11,136.000,38.0,0.487179,"[8, -16, 16, -24, 14]"
3,Stamina RPG 6/[11] [124] Glory Days,c0b5988bae418d86ccac7c7209f605932857bb81,11,124.000,44.0,0.483516,"[7, -17, 15, -29, 15, 7]"
4,Stamina RPG 6/[11] [145] Hawaiian Roller Coast...,d0b51e7437f6705f45bf6d9c1254964330bfcc4e,11,145.000,29.0,0.358025,"[1, -16, 10, -11, 1, -4, 4, -5, 1, 1, -8, 10, ..."
...,...,...,...,...,...,...,...
2000,kpoop/Blue Flame (hard),b1488049db7b70d7dea29c5698a9d8d5f4b5fe82,19,224.000,104.0,0.764706,"[32, -32, 72]"
2001,kpoop/Blue Flame,2b81028b58d210aa7fe66bea7f973360f95e8143,31,336.000,155.0,0.763547,"[48, -48, 107]"
2002,_misc/140BPM Stamina Killer,b9c0db86a2e1346db5967b712aeb1ff876154c28,21,140.000,8347.0,1.000000,[8347]
2003,_misc/[Marathon] Life Starts At 200 BPM,c2dcb36e047968bc2ece44ae23503e5b3b1e9bb9,23,199.752,4148.0,0.680446,"[6, -2, 6, -2, 7, 7, -9, 14, -2, 5, -3, 5, -42..."


In [11]:
df.to_json('../public/dataset.json', orient='records', index=False)